# InternVL3-8B with 8-bit Quantization for 4x V100 (16GB each)

In [ ]:
from pathlib import Path
import random
import math

import numpy as np
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

# Set random seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print("✅ Random seed set to 42 for reproducibility")

In [ ]:
# Update this path to your local InternVL3-8B model
model_id = "/path/to/InternVL3-8B"
# Update this path to your test image
imageName = "sample_data/synthetic_invoice_014.png"

print("🔧 Loading InternVL3-8B with 8-bit quantization for 4x V100 GPUs...")

# 8-bit quantization config for V100 memory efficiency
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

# Load model with 8-bit quantization and device_map
model = AutoModel.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",  # Required for quantization
    trust_remote_code=True
).eval()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    trust_remote_code=True, 
    use_fast=False
)

print("✅ Model and tokenizer loaded successfully with 8-bit quantization")
print(f"✅ Model distributed across devices: {model.hf_device_map}")

In [ ]:
# Official InternVL3 image preprocessing (from docs)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

# Process image - use float16 for 8-bit quantized models
print("🖼️  Processing image...")
pixel_values = load_image(imageName, max_num=12).to(torch.float16)

# Move to device where vision model is located
vision_device = model.vision_model.device if hasattr(model, 'vision_model') else 'cuda:0'
pixel_values = pixel_values.to(vision_device)

print(f"✅ Image processed: {pixel_values.shape}")
print(f"✅ Number of image tiles: {pixel_values.shape[0]}")
print(f"✅ Pixel values dtype: {pixel_values.dtype}")
print(f"✅ Pixel values on device: {vision_device}")

# Visual Question Answering - ask a simple question about the image
prompt = 'You are an expert document analyzer specializing in bank statement extraction. Extract the transaction data from this Australian bank statement.'
# InternVL3 format: <image>\n + question
formatted_question = f'<image>\n{prompt}'
print(f"❓ Question: {prompt}")

In [ ]:
# Deterministic generation config with multi-turn support
generation_config = dict(
    max_new_tokens=2000,
    do_sample=False  # Pure greedy decoding for deterministic output
)

# Clear CUDA cache before generation
torch.cuda.empty_cache()

# Generate response using InternVL3 API
print("🤖 Generating response with InternVL3-8B (8-bit quantized)...")
print(f"💾 GPU Memory before generation:")
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1e9
    reserved = torch.cuda.memory_reserved(i) / 1e9
    print(f"   GPU {i}: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

try:
    # With device_map, model is not wrapped - call chat() directly
    response, history = model.chat(
        tokenizer, 
        pixel_values, 
        formatted_question, 
        generation_config,
        history=None,
        return_history=True
    )
    
    print("✅ Response generated successfully!")
    print("\n" + "="*60)
    print("INTERNVL3 RESPONSE:")
    print("="*60)
    print(response)
    print("="*60)
    
except Exception as e:
    print(f"❌ Error during inference: {e}")
    print(f"Error type: {type(e).__name__}")
    import traceback
    traceback.print_exc()
finally:
    # Clean up GPU memory
    torch.cuda.empty_cache()

In [ ]:
# Optional: Save response to file
try:
    output_path = Path("internvl3_8b_quantized_vqa_output.txt")
    
    with output_path.open("w", encoding="utf-8") as text_file:
        text_file.write(response)
    
    print(f"✅ Response saved to: {output_path}")
    print(f"📄 File size: {output_path.stat().st_size} bytes")
    
except NameError:
    print("❌ Error: 'response' variable not defined.")
    print("💡 Please run the previous cell first to generate the response.")
    
except Exception as e:
    print(f"❌ Error saving file: {e}")

## Multi-Turn Conversation Example

InternVL3 supports multi-turn conversations using the history parameter:

In [ ]:
# Second turn conversation (uses history from first turn)
try:
    follow_up_question = "Can you provide the total amount from the document?"
    print(f"❓ Follow-up question: {follow_up_question}")
    
    # Use history from previous turn
    response2, history = model.chat(
        tokenizer,
        None,  # No new image for follow-up
        follow_up_question,
        generation_config,
        history=history,  # Pass previous history
        return_history=True
    )
    
    print("\n" + "="*60)
    print("FOLLOW-UP RESPONSE:")
    print("="*60)
    print(response2)
    print("="*60)
    
except NameError:
    print("❌ Error: 'history' variable not defined.")
    print("💡 Please run the first generation cell to create conversation history.")
except Exception as e:
    print(f"❌ Error during follow-up: {e}")

## Large Document Optimization Settings

For documents with abundant content, these optimized settings improve extraction accuracy by providing higher resolution and more detailed image processing:

In [ ]:
# OPTIMIZED SETTINGS FOR LARGE DOCUMENTS
# Use these settings when processing documents with abundant content

# HIGH RESOLUTION PROCESSING
# Increase input_size for better text clarity (default: 448)
high_res_input_size = 672  # 50% increase in resolution

# MAXIMUM TILE COVERAGE  
# Increase max_num for better content coverage (default: 12)
max_tiles_large_doc = 24  # Double the tiles for comprehensive coverage

# ENHANCED GENERATION CONFIG
# Optimized for longer, more detailed responses
large_doc_generation_config = dict(
    max_new_tokens=4000,        # Double the response length
    do_sample=False,            # Keep deterministic
    repetition_penalty=1.05,    # Reduce repetition in long responses
    length_penalty=1.0,         # Neutral length penalty
    num_beams=1,               # Greedy decoding for consistency
)

print("🚀 LARGE DOCUMENT OPTIMIZATION SETTINGS")
print(f"📐 Input resolution: {high_res_input_size}x{high_res_input_size} (vs default 448x448)")
print(f"🔲 Maximum tiles: {max_tiles_large_doc} (vs default 12)")
print(f"📝 Max response tokens: {large_doc_generation_config['max_new_tokens']} (vs default 2000)")
print(f"🔄 Repetition penalty: {large_doc_generation_config['repetition_penalty']}")
print()

# Process the same image with optimized settings
print("🖼️  Processing image with LARGE DOCUMENT settings...")
pixel_values_hires = load_image(
    imageName, 
    input_size=high_res_input_size,  # Higher resolution
    max_num=max_tiles_large_doc      # More tiles
).to(torch.float16)

# Move to device
pixel_values_hires = pixel_values_hires.to(vision_device)

print(f"✅ HIGH-RES Image processed: {pixel_values_hires.shape}")
print(f"✅ Number of image tiles: {pixel_values_hires.shape[0]} (vs {pixel_values.shape[0]} with default)")
print(f"✅ Resolution per tile: {high_res_input_size}x{high_res_input_size} (vs 448x448 default)")

# Calculate memory usage difference
default_pixels = pixel_values.numel()
hires_pixels = pixel_values_hires.numel()
memory_increase = (hires_pixels / default_pixels - 1) * 100

print(f"📊 Memory increase: +{memory_increase:.1f}% ({hires_pixels:,} vs {default_pixels:,} pixels)")
print(f"💾 Estimated VRAM increase: ~{(hires_pixels - default_pixels) * 2 / 1e9:.2f}GB")

In [ ]:
# GENERATE RESPONSE WITH LARGE DOCUMENT OPTIMIZATIONS
print("🤖 Generating response with LARGE DOCUMENT optimization settings...")

# Enhanced prompt for comprehensive extraction
comprehensive_prompt = """You are an expert document analyzer specializing in comprehensive bank statement extraction. 

Extract ALL transaction data from this Australian bank statement with maximum detail and accuracy. Include:
- Every transaction date, description, and amount
- Account balances after each transaction  
- Any fees, charges, or interest payments
- Beginning and ending balances
- Account details and statement period

Process the entire document systematically, ensuring no transactions are missed or abbreviated."""

formatted_question_comprehensive = f'<image>\n{comprehensive_prompt}'

print(f"💾 GPU Memory before large document generation:")
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1e9
    reserved = torch.cuda.memory_reserved(i) / 1e9
    print(f"   GPU {i}: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

try:
    # Generate with high-resolution image and enhanced settings
    response_hires, history_hires = model.chat(
        tokenizer, 
        pixel_values_hires,  # High-resolution processed image
        formatted_question_comprehensive,  # Enhanced prompt
        large_doc_generation_config,  # Optimized generation config
        history=None,
        return_history=True
    )
    
    print("✅ HIGH-RESOLUTION response generated successfully!")
    print(f"📊 Response length: {len(response_hires)} characters")
    print(f"📊 Token estimate: ~{len(response_hires.split())} words")
    
    print("\n" + "="*80)
    print("HIGH-RESOLUTION COMPREHENSIVE EXTRACTION:")
    print("="*80)
    print(response_hires)
    print("="*80)
    
    # Save high-res response
    hires_output_path = Path("internvl3_8b_quantized_HIRES_vqa_output.txt")
    with hires_output_path.open("w", encoding="utf-8") as text_file:
        text_file.write(response_hires)
    
    print(f"\n✅ High-resolution response saved to: {hires_output_path}")
    print(f"📄 File size: {hires_output_path.stat().st_size} bytes")
    
except Exception as e:
    print(f"❌ Error during high-resolution inference: {e}")
    print(f"Error type: {type(e).__name__}")
    import traceback
    traceback.print_exc()
finally:
    # Display final memory usage
    print(f"\n💾 GPU Memory after large document generation:")
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1e9
        reserved = torch.cuda.memory_reserved(i) / 1e9
        print(f"   GPU {i}: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
    
    torch.cuda.empty_cache()

## Additional Optimization Options

For even more aggressive optimization with abundant memory:

In [ ]:
# EXTREME OPTIMIZATION FOR VERY LARGE DOCUMENTS
# Only use if you have abundant GPU memory (32GB+ per GPU)

print("⚡ EXTREME LARGE DOCUMENT SETTINGS")
print("🚨 WARNING: These settings require significant GPU memory!")
print()

# Ultra-high resolution settings
ultra_input_size = 896        # Double default resolution
ultra_max_tiles = 36          # 3x default tiles  
ultra_tokens = 6000           # Extended response length

# Calculate memory requirements
def estimate_memory_usage(input_size, max_tiles):
    # Rough estimation: input_size^2 * max_tiles * 3 channels * 2 bytes (float16)
    pixels_per_tile = input_size * input_size * 3
    total_pixels = pixels_per_tile * max_tiles
    memory_gb = total_pixels * 2 / 1e9  # 2 bytes per float16
    return memory_gb

default_memory = estimate_memory_usage(448, 12)
high_res_memory = estimate_memory_usage(672, 24)
ultra_memory = estimate_memory_usage(ultra_input_size, ultra_max_tiles)

print(f"📊 MEMORY REQUIREMENTS COMPARISON:")
print(f"   Default (448px, 12 tiles):  ~{default_memory:.2f}GB")
print(f"   High-res (672px, 24 tiles): ~{high_res_memory:.2f}GB")
print(f"   Ultra (896px, 36 tiles):    ~{ultra_memory:.2f}GB")
print()

# Check available GPU memory
total_available = 0
for i in range(torch.cuda.device_count()):
    available = torch.cuda.get_device_properties(i).total_memory / 1e9
    allocated = torch.cuda.memory_allocated(i) / 1e9
    free = available - allocated
    total_available += free
    print(f"   GPU {i}: {free:.1f}GB free of {available:.1f}GB total")

print(f"   Total free memory: {total_available:.1f}GB")
print()

if total_available > ultra_memory + 5:  # 5GB buffer
    print("✅ Sufficient memory for ULTRA settings")
    
    ultra_generation_config = dict(
        max_new_tokens=ultra_tokens,
        do_sample=False,
        repetition_penalty=1.08,    # Stronger repetition control
        length_penalty=1.1,         # Encourage longer responses
        num_beams=1,
    )
    
    print(f"🎯 Ultra settings ready:")
    print(f"   Resolution: {ultra_input_size}x{ultra_input_size}")
    print(f"   Max tiles: {ultra_max_tiles}")
    print(f"   Max tokens: {ultra_tokens}")
    print()
    print("💡 To use ultra settings, run:")
    print(f"   pixel_values_ultra = load_image(imageName, input_size={ultra_input_size}, max_num={ultra_max_tiles})")
    print(f"   # Then use ultra_generation_config in model.chat()")
    
elif total_available > high_res_memory + 3:  # 3GB buffer
    print("✅ Sufficient memory for HIGH-RES settings (recommended)")
    print("⚠️  Not enough memory for ULTRA settings")
    
else:
    print("⚠️  Limited memory - stick with DEFAULT settings")
    print("💡 Consider reducing max_num tiles or input_size")

print()
print("🔧 OPTIMIZATION TIPS:")
print("1. Start with high-res settings (672px, 24 tiles)")
print("2. Monitor GPU memory usage during processing")
print("3. Increase resolution before increasing tile count")
print("4. Use comprehensive prompts for better extraction")
print("5. Save different resolution outputs for comparison")